# 4.5 代码预训练与专训 (Code Pretraining)

> 🕐 预估学习时间：40分钟

代码模型（CodeLlama、DeepSeek-Coder、Qwen2.5-Coder、StarCoder）在通用 LM 之上强调仓库级上下文、填充式目标、执行反馈与单测驱动。本节用可运行的小例子覆盖核心训练信号。

本节涵盖：
- 代码语料配比与去污
- Causal LM vs FIM（Fill-in-the-Middle）
- 仓库级上下文打包
- 执行反馈 / 单测奖励
- 评测：Pass@k


## 1. 代码语料与配比直觉

| 来源 | 作用 | 风险 |
|------|------|------|
| GitHub/The Stack | 广度 | 许可证、重复、密钥泄漏 |
| 文档/Notebook | 自然语言对齐 | 质量不齐 |
| 合成题解 | 覆盖算法模式 | 分布偏窄 |
| 内部仓 | 企业风格 | 保密与去标识 |

配比时常提高高质量、可运行子集权重，并做密钥/PII 扫描。


In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

SECRET_PAT = re.compile(r"(api[_-]?key\s*=\s*\S+|AKIA[0-9A-Z]{16})", re.I)


def clean_code(text: str) -> str | None:
    if SECRET_PAT.search(text):
        return None
    # strip giant minified lines
    lines = [ln for ln in text.splitlines() if len(ln) < 300]
    if len(lines) < 2:
        return None
    return '\n'.join(lines)


samples = [
    'def add(a,b):\n    return a+b\n',
    'api_key = "sk-secret-123"\nprint(api_key)\n',
    'x=' + 'a'*400 + '\n',
]
print('=== Code Cleaning ===')
for s in samples:
    out = clean_code(s)
    print(repr(s[:40]), '->', 'KEEP' if out else 'DROP')
print(f'Key: License/PII/secret scanning is mandatory before code pretraining.')



## 2. FIM：Fill-in-the-Middle

编辑器补全常需要根据前缀+后缀填中间。FIM 训练时随机切 `prefix/middle/suffix`，重排为 `prefix + suffix + middle`（或特殊 token 分隔）做自回归。


In [ ]:
def fim_pack(tokens, fim_prob=0.5):
    '''tokens: 1D LongTensor. Returns rearranged tokens for FIM or original.'''
    if torch.rand(()) > fim_prob or tokens.numel() < 6:
        return tokens, 'clm'
    n = tokens.numel()
    i = int(torch.randint(1, n - 2, (1,)))
    j = int(torch.randint(i + 1, n - 1, (1,)))
    prefix, middle, suffix = tokens[:i], tokens[i:j], tokens[j:]
    # special ids: 1=<fim_prefix>, 2=<fim_suffix>, 3=<fim_middle>
    packed = torch.cat([
        torch.tensor([1]), prefix,
        torch.tensor([2]), suffix,
        torch.tensor([3]), middle,
    ])
    return packed, 'fim'


class CodeLM(nn.Module):
    def __init__(self, vocab=100, d=64):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.block = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 4, 128, batch_first=True), 2
        )
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        h = self.embed(x)
        # causal mask
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        h = self.block(h, mask=mask)
        return self.head(h)


model = CodeLM()
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
print('=== FIM + CLM Joint Training ===')
for step in range(50):
    toks = torch.randint(4, 100, (16, 32))
    packed_batch = []
    modes = []
    for row in toks:
        p, mode = fim_pack(row)
        # pad/trim to 40
        if p.numel() < 40:
            p = F.pad(p, (0, 40 - p.numel()))
        else:
            p = p[:40]
        packed_batch.append(p)
        modes.append(mode)
    batch = torch.stack(packed_batch)
    logits = model(batch[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, 100), batch[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 49:
        fim_ratio = sum(m == 'fim' for m in modes) / len(modes)
        print(f'step={step:02d} loss={loss.item():.4f} fim_ratio={fim_ratio:.2f}')
print(f'\nKey: Interleave FIM with CLM so the model supports both chat and IDE fill-in.')


## 3. 仓库级上下文打包

把同仓库相关文件按依赖/路径优先级拼进窗口（或用 RAG 检索），让模型学习跨文件 API。教学实现用“主文件 + 依赖摘要”拼接。


In [ ]:
def pack_repo_context(files: dict[str, str], main: str, max_chars=300):
    '''files: path->content. Put imports of main first, then main.'''
    import_re = re.compile(r'^(?:from|import)\s+([\w\.]+)', re.M)
    deps = import_re.findall(files.get(main, ''))
    chunks = []
    for d in deps:
        # map module-ish name to file
        cand = d.replace('.', '/') + '.py'
        if cand in files:
            chunks.append(f'# file: {cand}\n' + files[cand])
    chunks.append(f'# file: {main}\n' + files[main])
    text = '\n\n'.join(chunks)
    return text[:max_chars]


repo = {
    'utils/math_utils.py': 'def add(a,b):\n    return a+b\n',
    'app/main.py': 'from utils.math_utils import add\n\ndef run():\n    return add(1,2)\n',
}
packed = pack_repo_context(repo, 'app/main.py')
print('=== Repo Packing ===')
print(packed)
print(f'\nKey: Cross-file packing teaches API usage beyond single-file pretraining.')


## 4. 执行反馈与 Pass@k

对可运行题：生成 k 个样本，在沙箱跑单测；Pass@k = 至少一发通过的比例。执行信号也可回灌为 RL/偏好数据。


In [ ]:
def pass_at_k(n, c, k):
    '''Canonical unbiased Pass@k estimator: n samples, c correct, choose k.'''
    if n - c < k:
        return 1.0
    return 1.0 - math.prod((n - c - i) / (n - i) for i in range(k))


import math

def run_tests(fn):
    tests = [((1, 2), 3), ((0, 0), 0), ((-1, 1), 0)]
    try:
        return all(fn(*a) == b for a, b in tests)
    except Exception:
        return False


candidates = [
    lambda a, b: a + b,
    lambda a, b: a - b,
    lambda a, b: a + b + 1,
    lambda a, b: a + b,
]
correct = sum(run_tests(fn) for fn in candidates)
n = len(candidates)
print('=== Execution Feedback / Pass@k ===')
print(f'correct={correct}/{n}')
for k in [1, 2, 4]:
    print(f'Pass@{k}={pass_at_k(n, correct, k):.3f}')
print(f'\nKey: Unit-test rewards are hard to hack and define the main code-model KPI.')


## 课后思考题

1. FIM 比例过高对对话/长生成有何副作用？如何调度？
2. 仓库级打包与 RAG 检索仓库，各自适合什么上下文长度预算？
3. Pass@k 很高但仓库真实修复率低，可能缺了什么训练信号？
4. 开源语料许可证不合规时，合成数据能替代到什么程度？

---
> 本节涵盖了4.5 代码预训练与专训的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
